# പാഠം 08 - മൾട്ടി-ഏജന്റ് ഡിസൈൻ മാതൃക


## സെറ്റ്‌അപ്പ്


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## എന്തുകൊണ്ട് മൾട്ടി-ഏജന്റ് സിസ്റ്റങ്ങൾ?

യാഥാർത്ഥ്യ ലോകത്തിലെ പ്രവർത്തനങ്ങൾ, യാത്രാ പദ്ധതി തയ്യാറാക്കൽ പോലുള്ളവ, നിരവധി വ്യത്യസ്ത പ്രകടനങ്ങളോട് ബന്ധപ്പെട്ടവയാണ് — ലോജിസ്റ്റിക്‌സ്, പ്രാദേശിക അറിവ്, ബജറ്റിംഗ്, മുതലായവ. എല്ലാം കൈകാര്യം ചെയ്യാൻ ശ്രമിക്കുന്ന ഏക ഏജന്റ് വേഗത്തിൽ കയ്യിൽ പിടിക്കാൻ ബുദ്ധിമുട്ടുള്ളതാക്കും.

മൾട്ടി-ഏജന്റ് സിസ്റ്റങ്ങൾ ഇത് **പ്രത്യേകത** മുഖേന പരിഹരിക്കുന്നു: ഓരോ ഏജന്റും ഒരു മേഖലയിലെ വിദഗ്ധതയിൽ ശ്രദ്ധ കേന്ദ്രീകരിക്കുന്നു, ഒരു പൊതുവായ വിദഗ്ധനേക്കാൾ ഉയർന്ന നിലവാരമുള്ള ഫലങ്ങൾ ലഭ്യമാക്കുന്നു. അവ **സ്കെയില്ബിലിറ്റിയും** മെച്ചപ്പെടുത്തുന്നു — പുതിയ ഏജന്റ്മാരെ (ഉദാഹരണത്തിന്, ഒരു ഫ്ലൈറ്റിൻ especialista, ഒരു റസ്റ്റോറന്റ് വിമർശകൻ) നിലവിലുള്ള വർക്ക്‌ഫ്ലോ മാറ്റുന്നതിനുമുൻപായി ചേർക്കാം. ഏജന്റ്മാർ ഘടനാപരമായ പൈപ്പ്ലൈനിലൂടെ ചേർത്ത് പ്രവര്‍ത്തിക്കുന്നു, ഒരുവിധത്തിൽ контекст് ഒരു ഏജന്റിൽ നിന്നും മറുതോറും ട്രാൻസ്ഫർ ചെയ്യുന്നു.


## പ്രത്യേക ഏജന്റുകള്‍ സൃഷ്ടിക്കുന്നത്


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ഒരു സീക്വൻഷ്യൽ വർക്‌ഫ്ലോ നിർമ്മാണം

`WorkflowBuilder` നിങ്ങൾക്ക് ഏജന്റുകളെ ഒരു ദിശാബദ്ധമായ ഗ്രാഫിലേക്ക് ബന്ധിപ്പിക്കാൻ അനുവദിക്കുന്നു. ഇവിടെ നാം ഒരു ലളിതമായ രണ്ട്-പടി പൈപ്പ്ലൈൻ സൃഷ്ടിക്കുന്നു: **TravelPlanner** യാത്രാപദ്ധതി തയ്യാറാക്കുന്നു, തുടർന്ന് **TravelConcierge** അത്.review ചെയ്ത് മെച്ചപ്പെടുത്തുന്നു.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## വർക്ക്‍ഫ്ലോയ്ക്ക് കൂടുതൽ ഏജന്റുകൾ ചേർക്കൽ

മൾട്ടി-എജന്റ് ഡീസൈൻ പാറ്റേണിന് വലിയ പ്രതിഫലങ്ങളിൽ ഒന്നാണ് അത് എളുപ്പത്തിൽ വിപുലീകരിക്കാൻ സാദ്ധ്യമാകുന്നത്. താഴെ ഞങ്ങൾ ഒരു **BudgetReviewer** ഏജന്റ് ചേർക്കുന്നു, ഇത് യാത്രക്കാരന്റെ ബഡ്ജറ്റിനോട് യോജിച്ചുള്ള പ്ലാനിനെ പരിശോധിച്ച്, ചിലവുകൾ പരിധി കടക്കാൻ ഇടയുള്ള ഇനങ്ങളെ ഫ്ലാഗ് ചെയ്യുകയും, പണം ലാഭിക്കുന്ന ബദലുകൾ പ്രস্তാവിക്കുകയും ചെയ്യുന്നവയാണ്. ഇനിയുള്ള വർക്ക്‍ഫ്ലോ മൂന്നുഎജന്റുകൾ കൃത്യക്രമം പ്രവർത്തിപ്പിക്കുന്നു: 

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## സംക്ഷേപം

ഈ പാഠത്തിൽ നിങ്ങൾക്ക് താഴെ പറയുന്ന കാര്യങ്ങൾ പഠിക്കാൻ കഴിഞ്ഞു:

1. **പ്രത്യേകജ്ഞയായ ഏജന്റുകൾ സൃഷ്ടിക്കുക** — ഓരോതും ഒരു ലക്ഷ്യമിട്ടു പ്രവർത്തിക്കുന്ന (പിൻവന്നു കാണൽ, കോണ്‍സിയോർജ്, ബജറ്റ് പരിശോധന).
2. **`WorkflowBuilder` ഉം `add_edge` ഉം ഉപയോഗിച്ച് ഏജന്റുകൾ ഒരു തുടർ പ്രവൃത്തി പ്രവാഹത്തിൽ ബന്ധിപ്പിക്കുക**.
3. **ഒരു ബഹുഏജന്റ് പൈപ്പ്‌ലൈൻ നിന്നും ഔട്ട്പുട്ട് സ്ട്രീം ചെയ്യുക**, ഏജന്റ് ആരാണെന്ന് പിന്‌ഗാമം ചെയ്യുക.
4. **പ്രവൃത്തി പ്രവാഹം വികസിപ്പിക്കുക** നിലവിലുള്ള ഏജന്റുകളെ മാറ്റാതെ പുതിയ ഏജന്റുകൾ ചൈനിൽ ചേർക്കുക.

ബഹുഏജന്റ് ഡിസൈൻ പാറ്റേൺ ഓരോ ഏജന്റിനെയും ലളിതമായി സൂക്ഷിച്ച്, ഏകദേശം ഏജന്റ് ഒറ്റക്ക് കൈകാര്യം ചെയ്യാനാകാത്ത സമൃദ്ധവും കൂടുതൽ വിശദമായ അവലോകന ഫലങ്ങളും ഉൽപ്പാദിപ്പിക്കുന്നു.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അസംബന്ധിത അറിയിപ്പ്**:  
ഈ രേഖ AI തർജമാ സേവനമായ [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് തർജമിച്ചവയാണ്. നമുക്ക് കൃത്യതക്കായി ശ്രമിച്ചിട്ടുണ്ടെങ്കിലും, സ്വയം പ്രവർത്തിക്കുന്ന തർജമയിൽ പിഴവുകളോ തെറ്റിച്ചൂകകളോ ഉണ്ടാകാവുന്നതാണ്. അവിടെയുള്ള ആദ്യഭാഷയിലെ യഥാർത്ഥ രേഖ ആണ് പ്രാമാണിക ഉറവിടം ആയി കണക്കാക്കേണ്ടത്. നിർണ്ണായക വിവരങ്ങളിലേക്കായി, പ്രൊഫഷണൽ മനുഷ്യ ترجم ഉപയോഗിച്ച് തർജമ ചെയ്യുന്നതാണ് ശുപാർശ ചെയ്യുന്നത്. ഈ തർജമയുടെ ഉപയോഗത്തിൽ ഉണ്ടാകുന്ന ഏതെങ്കിലും തെറ്റു മനസ്സിലാക്കലുകളോ തെറ്റിദ്ധാരണകളോ എനിക്കോ നമുക്കോ ഉത്തരവാദിത്വമില്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
